# Learning module 9 — Prepare the iOS session options locally

**Small milestone:** turn the selected run profile into an `XCUITestOptions` object and inspect the W3C session-request payload. This cell does not contact Appium or the Simulator.

The information comes from three places:

| Source | Requested capability | Meaning |
| --- | --- | --- |
| Profile's iOS platform selects `XCUITestOptions` | `platformName: iOS` | Platform to automate; supplied by this options class's defaults. |
| XCUITest adapter selection | `appium:automationName: XCUITest` | Registered automation driver; also supplied by the class defaults. |
| `selected_profile["simulator_udid"]` | `appium:udid` | Specific Simulator recorded in module 3. |
| `selected_profile["app_bundle_id"]` | `appium:bundleId` | Installed Movies application recorded in module 3. |
| Explicit startup policy for this exercise | `appium:noReset: true` | Request preservation of existing application state during session startup. |
| Explicit startup policy for this exercise | `appium:fullReset: false` | Do not request a full uninstall/reset. |

The reset policy is our stated learning-exercise choice; it is not extracted from the English scenario. It does not guarantee that the app or backend state will remain unchanged once a session and interactions occur. A real session may launch or reuse the target app. We identify the existing installation by bundle ID, without supplying an app artifact to install. [XCUITest capabilities](https://appium.github.io/appium-xcuitest-driver/latest/reference/capabilities/)

`XCUITestOptions` provides convenient Python properties such as `udid` and `bundle_id`. In the installed client, `to_w3c()` produces the namespaced capability map and the W3C request envelope. We use that serializer instead of manually inventing the protocol keys.

**Run this notebook's scenario/profile setup cells first.** Use the project's Python kernel with the client installed in module 8. Node/Appium PATH activation is not needed to construct this local Python object.

## This notebook

Part **09 — Prepare requested iOS session capabilities** of the learning sequence. [Notebook index](README.md).

Select the project's `.venv` Python kernel and run this notebook from top to bottom. Its Python setup is included locally; running another notebook in another kernel does not populate this one's variables or PATH. The supported working directories are the project root, `server/`, and `server/sample_scenarios/`.

**Execution scope:** Constructs local options; no connection.

Saved lesson outputs and embedded screenshots are historical snapshots preserved during the split on November 3, 2026. Newly included setup cells have no saved outputs. Rerunning an executable scenario creates fresh local artifacts; these continue living in `server/.tools/`, not beside the notebooks.

### Setup for this notebook

These source-only prerequisite cells are repeated intentionally so the lesson is runnable in a fresh kernel. They configure local values/environment and inspect tool versions; they do not launch an Appium server, create a session, or tap/type in the app. The original lessons remain linked for explanation.


In [1]:
# Setup: Sample scenario input (lesson 01)
from pprint import pprint

scenario_text = (
    "In Movies, search for Interstellar, open the 2014 movie, "
    "and verify that its details show the title Interstellar "
    "and release year 2014."
)

scenario_input = {
    "instruction": scenario_text,
    "application_name": "Movies",
    "application_profile_ref": None,
    "requested_actions": [
        "search_for_movie",
        "open_matching_movie_details",
    ],
    "search_query": "Interstellar",
    "target_item_identity": {"title": "Interstellar", "release_year": 2014},
    "expected_title": "Interstellar",
    "expected_release_year": 2014,
    "title_matching_rule": "exact",  # Proposed comparison rule for this exercise.
}

pprint(scenario_input, sort_dicts=False)


{'instruction': 'In Movies, search for Interstellar, open the 2014 movie, and '
                'verify that its details show the title Interstellar and '
                'release year 2014.',
 'application_name': 'Movies',
 'application_profile_ref': None,
 'requested_actions': ['search_for_movie', 'open_matching_movie_details'],
 'search_query': 'Interstellar',
 'target_item_identity': {'title': 'Interstellar', 'release_year': 2014},
 'expected_title': 'Interstellar',
 'expected_release_year': 2014,
 'title_matching_rule': 'exact'}


In [2]:
# Setup: Recorded run profile (lesson 03)
from pprint import pprint

run_profile = {
    "id": "movies-ios-local",
    "platform": "ios",
    "execution_provider": "local",
    "app_bundle_id": "com.example.movies",
    "simulator_udid": "4F0707AE-45BC-464D-9099-6EE76C3740A9",
}

app_profiles = {run_profile["id"]: run_profile}
scenario_input["application_profile_ref"] = run_profile["id"]
selected_profile = app_profiles[scenario_input["application_profile_ref"]]

print("Scenario profile reference:", scenario_input["application_profile_ref"])
pprint(selected_profile, sort_dicts=False)


Scenario profile reference: movies-ios-local
{'id': 'movies-ios-local',
 'platform': 'ios',
 'execution_provider': 'local',
 'app_bundle_id': 'com.example.movies',
 'simulator_udid': '4F0707AE-45BC-464D-9099-6EE76C3740A9'}


In [3]:
import json
from appium.options.ios import XCUITestOptions

if "selected_profile" not in globals():
    raise RuntimeError("Run the scenario/profile setup cells in this notebook first.")
if selected_profile.get("platform") != "ios":
    raise ValueError("This example requires an iOS run profile.")
for field in ["app_bundle_id", "simulator_udid"]:
    value = selected_profile.get(field)
    if not isinstance(value, str) or not value.strip():
        raise ValueError(f"The selected profile needs a nonempty {field}.")

ios_options = XCUITestOptions()
ios_options.udid = selected_profile["simulator_udid"]
ios_options.bundle_id = selected_profile["app_bundle_id"]
ios_options.no_reset = True
ios_options.full_reset = False

session_request = ios_options.to_w3c()
print(json.dumps(session_request, indent=2))


{
  "capabilities": {
    "firstMatch": [
      {}
    ],
    "alwaysMatch": {
      "appium:automationName": "XCUITest",
      "platformName": "iOS",
      "appium:udid": "4F0707AE-45BC-464D-9099-6EE76C3740A9",
      "appium:bundleId": "com.example.movies",
      "appium:noReset": true,
      "appium:fullReset": false
    }
  }
}


### Read the requested capabilities, then pause

`alwaysMatch` contains the requested platform, driver, target, and startup settings. The empty entry in `firstMatch` adds no alternative settings. The SDK has prepared this locally; it has not sent a request or received accepted capabilities from Appium.

Notice the distinction between `platformName`, a standard capability, and the Appium-specific keys with the `appium:` prefix. The installed SDK normalizes that distinction in `to_w3c()`.

The following information is intentionally separate:

| Information | Where it belongs |
| --- | --- |
| Appium server URL | Client connection configuration: where to send the session request. |
| Scenario text, search query, and expected title/year | Test intent and assertions used by the agents. |
| Semantics/accessibility locators | Actions grounded in later UI observations. |
| Profile ID and execution provider | Our system's profile lookup and routing context. |
| Actual session ID and accepted capabilities | The server response after a session is successfully created. None exists here. |

The UDID and bundle ID are the recorded target snapshot, not a fresh installation/device check. Revalidate the target before a real session. Keep the locally requested settings separate from later server responses and UI evidence.

These are Python objects in the current notebook kernel. They are not yet a LangGraph run, a checkpoint, or a test result.

**Pause here:** trace profile → options object → serialized request. We have prepared the configuration only. The next operational milestone is a local Appium server/status check, after following up the recorded Node dependency audit findings; actual device-session creation remains a separate step.

[Index](README.md) · [Previous lesson](08_python_client.ipynb) · [Next lesson](10_appium_server_status.ipynb)